# Who Wants to be a PoliMillionaire? — chatbot investigation

**Course:** Natural Language Processing, Politecnico di Milano, AY 2025/26  
**Due:** 2 June 2026, 23:00 (WeBeep)

| Name | Email | GitHub | PoliMillionaire user |
|---|---|---|---|
| Leon Fuß | leon.fuss@icloud.com | leonfuss | leonfuss |
| Antoine Gaborieau | _ | AntoineGaborieau | _ |
| Aleksa Pulai | _ | aleksapulai | _ |
| Luca Ilardi | luca7203@icloud.com | LucBruc | LucBruc |
| _ | _ | _ | _ |

**Video:** <link to be added before submission>

**Coding assistants used:** _fill in before submission — which tools, what for, and a line confirming the whole assignment wasn't handed to an LLM._

## Introduction

Four-category MCQ trivia. Up to 15 questions per game, 30 s server timer per question, one wrong answer ends the run. Competitions: Entertainment (0), Ancient History + Politics (1), Science + Nature (2), Maths (3).

We tested four strategies: zero-shot LLM, `calc_react` (ReAct loop + sympy calculator), `rag_calc_react` (calc-react with retrieved MATH exemplars), and `auto` — a router that dispatches Wikipedia-RAG to 0/1/2 and `rag_calc_react` to 3. They all return the same `AnswerDecision` and log to the same SQLite DB, so the same corpus replays against any of them.

The notebook reports accuracy by (strategy, model, competition), prompt sensitivity, latency, and a wiki_rag ablation. Everything below is offline replay; the live-play cell at the end is the only one that hits the server.


## Setup

Colab-only. Clones the repo, installs deps, mounts Drive. Skip if you're running locally. Needs `GH_TOKEN`, `POLIMILLIONAIRE_API_URL`, `POLIMILLIONAIRE_USER`, `POLIMILLIONAIRE_PASSWORD` in Colab Secrets.


In [ ]:
# --- Colab-only bootstrap: skip this cell when running locally ---
import os
import sys

from google.colab import drive, userdata

gh_token = userdata.get("GH_TOKEN")
repo_dir = "/content/polimillionaire"

if os.path.isdir(repo_dir):
    os.chdir(repo_dir)
    !git checkout -q main && git pull --ff-only origin main
else:
    !git clone https://{gh_token}@github.com/leonfuss/polimillionaire.git $repo_dir
    os.chdir(repo_dir)

# CUDA wheel index for llama-cpp-python's pre-built T4 wheel.
!pip install -q -r requirements-colab.txt \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 \
    && pip install -q -e . --no-deps

if "/content/polimillionaire/src" not in sys.path:
    sys.path.insert(0, "/content/polimillionaire/src")

# Mount Drive and point the DB path at the shared question log.
drive.mount("/content/drive")
INDEX_ROOT = "/content/drive/MyDrive/PoliMillionaire"
os.environ.setdefault(
    "POLIMILLIONAIRE_DB_PATH",
    f"{INDEX_ROOT}/questions.sqlite",
)

# Symlink Drive's pre-built FAISS/BM25 index into the repo so the factory
# can find it at data/index/ without duplicating the ~1 GB onto the VM.
index_link = "/content/polimillionaire/data/index"
if not os.path.exists(index_link):
    os.makedirs("/content/polimillionaire/data", exist_ok=True)
    os.symlink(f"{INDEX_ROOT}/index", index_link)

print("DB path:", os.environ["POLIMILLIONAIRE_DB_PATH"])

## Load shared resources

Load the LLM once. `load_llm` caches by alias so re-running this cell in the same session is free.


In [ ]:
import os
from pathlib import Path

from polimillionaire import load_llm

llm = load_llm("qwen3-8b")
DB_PATH = Path(os.environ.get("POLIMILLIONAIRE_DB_PATH", "data/questions.sqlite"))
print("DB path:", DB_PATH, "| exists:", DB_PATH.exists())

## Manual baseline

The labelled corpus comes from us playing manually before any strategy was ready. Every question the server showed us, with the answer it confirmed, sits in `data/questions.sqlite` — about 234 rows. That's what offline replay runs against.


## Offline replay infrastructure

`replay_records` runs the strategy over every labelled question and returns one record per row. `to_polars` makes it a DataFrame. The rest of the notebook is variations on this loop.


In [ ]:
import polars as pl

from polimillionaire.eval.replay import replay_records
from polimillionaire.eval.results import to_polars
from polimillionaire.strategies import make_strategy

## Leaderboard

Replay several `(strategy, model)` combinations and tabulate accuracy by competition. Add entries to `STRATEGIES_TO_COMPARE` as more strategies land.


In [ ]:
STRATEGIES_TO_COMPARE = [
    ("zero_shot", "qwen3-8b"),
    # ("calc_react", "qwen3-8b"),
    # ("auto", "qwen3-8b"),
]

frames = []
for strategy_name, model_alias in STRATEGIES_TO_COMPARE:
    _llm = load_llm(model_alias)
    strategy = make_strategy(strategy_name, _llm)
    records = replay_records(strategy, DB_PATH, show_progress=True)
    frames.append(to_polars(records))

df = pl.concat(frames) if frames else pl.DataFrame()

if not df.is_empty():
    df.group_by(["strategy_name", "model_name", "competition_id"]).agg(
        pl.col("correct").mean().alias("accuracy"),
        pl.col("correct").count().alias("n"),
    ).sort(["strategy_name", "competition_id"])

## Accuracy by competition

Same data, pivoted: one row per `(strategy, model)`, one column per competition. Easier to see which categories the retrieval actually helps with.


In [ ]:
if not df.is_empty():
    df.group_by(["strategy_name", "model_name", "competition_id"]).agg(
        pl.col("correct").mean().alias("accuracy"),
    ).pivot(
        on="competition_id",
        index=["strategy_name", "model_name"],
        values="accuracy",
    ).sort("strategy_name")

## Latency vs accuracy

Each strategy costs different wall-clock time per question. Plotting accuracy against mean latency shows whether the slow ones earn their keep against the 30 s server timer.


In [ ]:
import matplotlib.pyplot as plt
from tueplots import bundles

plt.rcParams.update(bundles.icml2024())

if not df.is_empty():
    agg = df.group_by(["strategy_name", "model_name"]).agg(
        pl.col("correct").mean().alias("accuracy"),
        pl.col("latency_ms").mean().alias("mean_latency_ms"),
    )

    fig, ax = plt.subplots()
    for row in agg.iter_rows(named=True):
        ax.scatter(row["mean_latency_ms"], row["accuracy"], label=row["strategy_name"])
        ax.annotate(row["strategy_name"], (row["mean_latency_ms"], row["accuracy"]), fontsize=6)
    ax.set_xlabel("mean latency (ms)")
    ax.set_ylabel("accuracy")
    ax.legend(fontsize=6)
    plt.tight_layout()
    plt.show()

## Ablation: wiki_rag retrieval components

wiki_rag has three knobs: dense retriever, BM25, cross-encoder reranker. The four interesting combinations are run against one competition to see which component does most of the work. `strict=True` skips configs where the index isn't built — without it we'd silently get four identical zero-shot rows.


In [ ]:
ABLATION_COMPETITION_ID = 2  # Science and Nature; change as needed

ablation_configs = [
    ({"use_dense": True, "use_sparse": True, "use_reranker": True}, "dense+sparse+rerank"),
    ({"use_dense": True, "use_sparse": True, "use_reranker": False}, "dense+sparse"),
    ({"use_dense": True, "use_sparse": False, "use_reranker": False}, "dense-only"),
    ({"use_dense": False, "use_sparse": True, "use_reranker": False}, "sparse-only"),
]

ablation_frames = []
for cfg, label in ablation_configs:
    try:
        # strict=True makes the factory raise if the wiki index is missing,
        # so we don't silently compare four identical zero-shot runs.
        s = make_strategy(
            "wiki_rag",
            llm,
            competition_id=ABLATION_COMPETITION_ID,
            strict=True,
            **cfg,
        )
    except FileNotFoundError as e:
        print(f"skipping ablation: {e}")
        break
    records = replay_records(
        s,
        DB_PATH,
        competition_id=ABLATION_COMPETITION_ID,
        show_progress=True,
    )
    frame = to_polars(records).with_columns(pl.lit(label).alias("ablation"))
    ablation_frames.append(frame)

ablation_df = pl.concat(ablation_frames) if ablation_frames else pl.DataFrame()
if not ablation_df.is_empty():
    ablation_df.group_by("ablation").agg(
        pl.col("correct").mean().alias("accuracy"),
        pl.col("correct").count().alias("n"),
    ).sort("accuracy", descending=True)

## Prompt sensitivity

Hold strategy + model constant, sweep `prompt_version`. Tells us whether a prompt edit actually moved the needle.


In [ ]:
PROMPT_STRATEGY = "zero_shot"
PROMPT_VERSIONS = ["v1"]  # extend as new versions land

prompt_frames = []
for pv in PROMPT_VERSIONS:
    s = make_strategy(PROMPT_STRATEGY, llm, prompt_version=pv)
    records = replay_records(s, DB_PATH, show_progress=True)
    prompt_frames.append(to_polars(records))

prompt_df = pl.concat(prompt_frames) if prompt_frames else pl.DataFrame()

if not prompt_df.is_empty():
    prompt_df.group_by("prompt_version").agg(
        pl.col("correct").mean().alias("accuracy"),
        pl.col("correct").count().alias("n"),
    ).sort("prompt_version")

## Model comparison

Hold strategy + prompt constant, swap models. Only the LLM changes.


In [ ]:
MODEL_STRATEGY = "zero_shot"
MODELS_TO_COMPARE = ["qwen3-8b"]  # extend with other entries from MODELS

model_frames = []
for alias in MODELS_TO_COMPARE:
    _llm = load_llm(alias)
    s = make_strategy(MODEL_STRATEGY, _llm)
    records = replay_records(s, DB_PATH, show_progress=True)
    model_frames.append(to_polars(records))

model_df = pl.concat(model_frames) if model_frames else pl.DataFrame()

if not model_df.is_empty():
    model_df.group_by(["model_name", "strategy_name"]).agg(
        pl.col("correct").mean().alias("accuracy"),
        pl.col("correct").count().alias("n"),
    ).sort("accuracy", descending=True)

## Qualitative trace

Pick one question and read the strategy's rationale. Worth doing when aggregate numbers look off — the failure mode is usually obvious in two or three rationales.


In [ ]:
import json
import sqlite3

from polimillionaire._vendor.millionaire_client.models import Option, Question
from polimillionaire.strategies import ZeroShotStrategy
from polimillionaire.strategies.base import Context

TRACE_QUESTION_ID = None  # set to an integer question_id from the DB

if TRACE_QUESTION_ID is not None and DB_PATH.exists():
    with sqlite3.connect(DB_PATH) as con:
        con.row_factory = sqlite3.Row
        row = con.execute(
            """
            SELECT question_id, question_text, options_json, level, competition_id,
                   correct_option_id_if_known
            FROM predictions
            WHERE question_id = ? AND correct_option_id_if_known IS NOT NULL
            LIMIT 1
            """,
            (TRACE_QUESTION_ID,),
        ).fetchone()

    if row:
        options = [Option(**o) for o in json.loads(row["options_json"])]
        q = Question(
            id=row["question_id"],
            text=row["question_text"],
            options=options,
            level=row["level"],
        )
        ctx = Context(competition_id=row["competition_id"], level=row["level"])
        trace_strategy = ZeroShotStrategy(llm)  # swap for any strategy
        decision = trace_strategy(q, ctx)

        print(f"Q: {q.text}")
        for opt in q.options:
            marker = "*" if opt.id == row["correct_option_id_if_known"] else " "
            chosen = "->" if opt.id == decision.option_id else "  "
            print(f"  {chosen}[{marker}] [{opt.id}] {opt.text}")
        print(f"\nrationale: {decision.rationale}")
        print(f"confidence: {decision.confidence}  latency: {decision.latency_ms} ms")
    else:
        print(f"question_id {TRACE_QUESTION_ID} not found in DB")
else:
    print("Set TRACE_QUESTION_ID to an integer to run this cell.")

## Live play

Plays an actual game against the server. Edit the constants and set `RUN_LIVE = True`. Needs credentials and writes to the shared log, so the guard is there to keep "Run All" from playing a real game.


In [ ]:
# --- live play parameters ----------------------------------------------------

RUN_LIVE = False  # flip to True to actually play
PLAY_COMPETITION_ID = 0  # 0=Entertainment, 1=AncientHist+Politics, 2=Science+Nature, 3=Maths
PLAY_STRATEGY = "auto"  # "auto" | "wiki_rag" | "rag_calc_react" | "calc_react" | "zero_shot"
PLAY_MODEL = "qwen3-8b"  # any alias from polimillionaire.MODELS
PLAY_MAX_GAMES = 1  # play this many back-to-back games against the same competition
PLAY_VERBOSE = True  # print retrieval hits + per-question reasoning

# Strategy-specific knobs; the factory forwards only the ones that apply.
PLAY_MAX_STEPS = 3  # calc_react / rag_calc_react: ReAct loop cap
PLAY_RAG_K = 3  # rag_calc_react: retrieved math exemplars
PLAY_STRICT = False  # wiki_rag / rag_calc_react: raise if the index is missing

# -----------------------------------------------------------------------------

if RUN_LIVE:
    from polimillionaire import load_llm, make_client
    from polimillionaire.play import auto_play_loop
    from polimillionaire.strategies import make_strategy

    play_llm = load_llm(PLAY_MODEL)
    play_strategy = make_strategy(
        PLAY_STRATEGY,
        play_llm,
        competition_id=PLAY_COMPETITION_ID,
        verbose=PLAY_VERBOSE,
        max_steps=PLAY_MAX_STEPS,
        k=PLAY_RAG_K,
        strict=PLAY_STRICT,
    )

    print(
        f"strategy={PLAY_STRATEGY} (effective: {play_strategy.strategy_name}), "
        f"model={PLAY_MODEL}, competition={PLAY_COMPETITION_ID}, "
        f"max_games={PLAY_MAX_GAMES}"
    )

    result = auto_play_loop(
        make_client(),
        competition_id=PLAY_COMPETITION_ID,
        strategy=play_strategy,
        max_games=PLAY_MAX_GAMES,
    )
    print(f"\nfinal: {result}")
else:
    print("RUN_LIVE is False -- skipping live play")

## Conclusions

- _placeholder: overall accuracy of the best strategy vs zero-shot baseline_
- _placeholder: which competition benefited most from retrieval augmentation_
- _placeholder: latency budget — which strategies are viable under the 30 s timer_
- _placeholder: prompt sensitivity findings_
- _placeholder: failure modes observed in the qualitative trace_